# Crys-JEPA · DFT Superconductivity Ablation — Google Colab

Ce notebook est **entièrement auto-contenu** : il installe les dépendances, clone le repo, télécharge les données, entraîne et évalue deux modèles.

| Mode | Description |
|------|-------------|
| `dft` | MLP sur features DFT uniquement (baseline) |
| `crys_jepa_dft` | Fusion tardive encoder structurel + DFT |

**Prérequis Colab** : activer un GPU via *Runtime → Change runtime type → T4 GPU* (gratuit).

---
**⚠️ Ordre d'exécution recommandé** : *Runtime → Run all*  
Ou exécuter les sections dans l'ordre : Setup → Données → Config → Tests → Entraînement → Évaluation.

## 0 · Setup

In [ ]:
# 0a — Vérification GPU
import subprocess, sys, os

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(gpu_info.stdout[:600])
    DEVICE = 'cuda'
else:
    print('Aucun GPU NVIDIA détecté — entraînement sur CPU (beaucoup plus lent).')
    DEVICE = 'cpu'

print('Python :', sys.version.split()[0])
print('Device  :', DEVICE)

In [ ]:
# 0b — [Optionnel] Monter Google Drive pour sauvegarder les checkpoints
# Passer USE_GDRIVE = True pour activer.
USE_GDRIVE = False
GDRIVE_CKPT_DIR = '/content/drive/MyDrive/crys_jepa_checkpoints'

if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_CKPT_DIR, exist_ok=True)
    print('Drive monté —', GDRIVE_CKPT_DIR)
else:
    print('Google Drive non monté (les checkpoints seront dans /content/Crys_JEPA/checkpoints/).')

In [ ]:
# 0c-i — Installation PyTorch 2.6.0 + CUDA 12.4
# Colab préinstalle PyTorch mais pas forcément la bonne version.
# Cette cellule réinstalle si nécessaire et indique s'il faut redémarrer le runtime.

TORCH_TARGET = '2.6.0'

try:
    import torch as _t
    current = _t.__version__.split('+')[0]
except ImportError:
    current = ''

if current != TORCH_TARGET:
    print(f'PyTorch actuel : {current!r} — installation de {TORCH_TARGET}+cu124...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'torch=={TORCH_TARGET}',
        'torchvision==0.21.0',
        'torchaudio==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124',
    ], check=True)
    print('\n⚠️  PyTorch réinstallé. Clique sur Runtime → Restart runtime, puis relance à partir de la cellule 0c-ii.')
    raise SystemExit('Restart required after PyTorch reinstall.')
else:
    import torch
    print(f'PyTorch {torch.__version__} déjà présent — OK.')

In [ ]:
# 0c-ii — Installation des dépendances du projet
# (à relancer après un éventuel redémarrage de runtime)

deps = [
    'torch-geometric',
    'ase==3.25.0',
    'easydict',
    'einops',
    'huggingface-hub',
    'lmdb==1.6.2',
    'matminer==0.9.3',
    'p-tqdm',
    'pandas',
    'pymatgen==2025.6.14',
    'pytest',
    'pyyaml',
    'scipy==1.16.2',
    'smact==3.2.0',
    'tqdm',
    'wandb',
]

print('Installation des dépendances...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps, check=True)
print('Dépendances installées.')

import torch
print(f'torch {torch.__version__} | CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')

In [ ]:
# 0c-iii — Cloner le repo Crys-JEPA
import os
from pathlib import Path

REPO_URL    = 'https://github.com/Baptistecaille/Crys_JEPA.git'
REPO_BRANCH = 'main'
REPO_DIR    = Path('/content/Crys_JEPA')

if REPO_DIR.exists():
    print('Repo déjà présent — mise à jour...')
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
else:
    print(f'Clonage de {REPO_URL} (branche {REPO_BRANCH})...')
    subprocess.run([
        'git', 'clone', '--depth', '1',
        '--branch', REPO_BRANCH,
        REPO_URL, str(REPO_DIR)
    ], check=True)

# Ajouter la racine du projet au PYTHONPATH
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)
print('Répertoire de travail :', Path.cwd())

In [ ]:
# 0d — Variables de configuration (modifier ici si besoin)
import torch
from pathlib import Path

PROJECT_DIR   = Path('/content/Crys_JEPA')
CSV_PATH      = PROJECT_DIR / 'data/raw/3DSC_MP.csv'
CIF_DIR       = PROJECT_DIR / 'data/raw/cifs'

# Appareil d'entraînement
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP       = DEVICE == 'cuda'   # Mixed-precision uniquement sur CUDA

# Hyperparamètres — ajuster selon le quota GPU disponible
EPOCHS        = 50
BATCH_SIZE    = 64    # T4 : 64 confortable ; A100 : 128+
LEARNING_RATE = 1e-4
NUM_WORKERS   = 2

# Répertoires de checkpoints (remplacer par GDRIVE_CKPT_DIR si Drive monté)
CKPT_DFT      = str(PROJECT_DIR / 'checkpoints/colab_dft')
CKPT_JEPA     = str(PROJECT_DIR / 'checkpoints/colab_crys_jepa_dft')

print(f'device={DEVICE}  AMP={USE_AMP}  epochs={EPOCHS}  batch={BATCH_SIZE}')
print(f'checkpoints DFT      : {CKPT_DFT}')
print(f'checkpoints JEPA+DFT : {CKPT_JEPA}')

## 1 · Téléchargement des données 3DSC

In [ ]:
# Télécharge le CSV + les CIFs depuis aimat-lab/3DSC (GitHub)
# ~10 000 structures · ~17 Mo total

import urllib.request, zipfile

n_cifs = len(list(CIF_DIR.glob('*.cif'))) if CIF_DIR.exists() else 0

if CSV_PATH.exists() and n_cifs > 0:
    print(f'Données déjà présentes — CSV: {CSV_PATH.name}, CIFs: {n_cifs}')
else:
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    CIF_DIR.mkdir(parents=True, exist_ok=True)

    if not CSV_PATH.exists():
        CSV_URL = (
            'https://raw.githubusercontent.com/aimat-lab/3DSC/main/'
            'superconductors_3D/data/final/MP/3DSC_MP.csv'
        )
        print('Téléchargement du CSV...')
        urllib.request.urlretrieve(CSV_URL, CSV_PATH)
        print(f'  CSV: {CSV_PATH.stat().st_size // 1024} Ko')
    else:
        print('CSV déjà présent.')

    if n_cifs == 0:
        ZIP_URL  = 'https://github.com/aimat-lab/3DSC/archive/refs/heads/main.zip'
        zip_path = PROJECT_DIR / 'data/raw/_3DSC_repo.zip'
        print('Téléchargement des CIFs (~17 Mo)...')
        urllib.request.urlretrieve(ZIP_URL, zip_path)
        print(f'  Archive: {zip_path.stat().st_size // (1024**2)} Mo')

        CIFS_MARKER = 'superconductors_3D/data/final/MP/cifs/'
        print('Extraction des CIFs...')
        n_extracted = 0
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for entry in zf.namelist():
                if CIFS_MARKER in entry and entry.endswith('.cif'):
                    target = CIF_DIR / Path(entry).name
                    with zf.open(entry) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
                    n_extracted += 1
        zip_path.unlink()
        print(f'  {n_extracted} CIFs extraits dans {CIF_DIR}')
    else:
        print(f'CIFs déjà présents ({n_cifs}).')

print()
print('CSV  :', CSV_PATH.exists(), '—', CSV_PATH if CSV_PATH.exists() else 'MANQUANT')
print('CIFs :', len(list(CIF_DIR.glob('*.cif'))), 'fichiers')

## 2 · Patch des configurations YAML

In [ ]:
# Met à jour les fichiers YAML avec les paramètres Colab
import yaml

CONFIGS = {
    'dft':           PROJECT_DIR / 'configs/ablation/dft.yaml',
    'crys_jepa_dft': PROJECT_DIR / 'configs/ablation/crys_jepa_dft.yaml',
}

for name, cfg_path in CONFIGS.items():
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)

    cfg['data']['csv_path']              = str(CSV_PATH)
    cfg['data']['cif_dir']               = str(CIF_DIR)
    cfg['training']['epochs']            = int(EPOCHS)
    cfg['training']['batch_size']        = int(BATCH_SIZE)
    cfg['training']['learning_rate']     = float(LEARNING_RATE)
    cfg['training']['device']            = DEVICE
    cfg['training']['use_amp']           = bool(USE_AMP)
    cfg['training']['num_workers']       = int(NUM_WORKERS)
    cfg['training']['pin_memory']        = DEVICE == 'cuda'
    cfg['checkpoints']['save_dir']       = CKPT_DFT if name == 'dft' else CKPT_JEPA

    with cfg_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    print(f'{name}:')
    print(f'  input_mode : {cfg["model"]["input_mode"]}')
    print(f'  device     : {cfg["training"]["device"]}  AMP={cfg["training"]["use_amp"]}')
    print(f'  save_dir   : {cfg["checkpoints"]["save_dir"]}')
    print()

## 3 · Tests rapides

In [ ]:
# Valide les imports et la lecture du dataset avant l'entraînement
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_superconductivity_mvp.py', '-q', '--tb=short'],
    capture_output=True, text=True, cwd=PROJECT_DIR
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Tests échoués — vérifier les erreurs ci-dessus avant de continuer.')

## 4 · Entraînement

> Les deux modèles sont entraînés directement via l'API Python pour afficher la progression en temps réel.  
> Sur un GPU T4 avec 50 époques, compter ~10–15 min par modèle.

In [ ]:
# Imports du pipeline d'entraînement
import copy
import torch
from src.training.train import load_config, train_from_config

print('Imports OK')
print(f'Utilisation de : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# 4a — Entraînement DFT baseline
print('=' * 60)
print('Mode : dft (MLP sur features DFT uniquement)')
print('=' * 60)

cfg_dft    = load_config(PROJECT_DIR / 'configs/ablation/dft.yaml')
result_dft = train_from_config(cfg_dft)

print()
print('Checkpoint :', result_dft['best_checkpoint'])
print('Test metrics (DFT) :')
for k, v in result_dft['test_metrics']['classification'].items():
    print(f'  cls/{k:12s} = {v:.4f}')
for k, v in result_dft['test_metrics']['regression'].items():
    print(f'  reg/{k:12s} = {v:.4f}')

In [ ]:
# 4b — Entraînement DFT-JEPA (fusion tardive encodeur structurel + DFT)
print('=' * 60)
print('Mode : crys_jepa_dft (encodeur structurel + features DFT)')
print('=' * 60)

cfg_jepa    = load_config(PROJECT_DIR / 'configs/ablation/crys_jepa_dft.yaml')
result_jepa = train_from_config(cfg_jepa)

print()
print('Checkpoint :', result_jepa['best_checkpoint'])
print('Test metrics (DFT-JEPA) :')
for k, v in result_jepa['test_metrics']['classification'].items():
    print(f'  cls/{k:12s} = {v:.4f}')
for k, v in result_jepa['test_metrics']['regression'].items():
    print(f'  reg/{k:12s} = {v:.4f}')

## 5 · Évaluation & tableau comparatif

In [ ]:
# Évalue les checkpoints sauvegardés et produit un CSV de prédictions
from src.training.run_evaluate import evaluate_checkpoint

RUNS = [
    ('dft',           CKPT_DFT  + '/best.pt', 'predictions_colab_dft.csv'),
    ('crys_jepa_dft', CKPT_JEPA + '/best.pt', 'predictions_colab_crys_jepa_dft.csv'),
]

eval_results = {}
for mode, ckpt_path, pred_csv in RUNS:
    cfg_path = PROJECT_DIR / f'configs/ablation/{mode}.yaml'
    print(f'Évaluation {mode}...')
    metrics = evaluate_checkpoint(
        cfg_path,
        ckpt_path,
        predictions_csv=PROJECT_DIR / pred_csv,
    )
    eval_results[mode] = metrics
    print(f'  F1={metrics["classification"]["f1"]:.4f}  '
          f'ROC-AUC={metrics["classification"]["roc_auc"]:.4f}  '
          f'MAE={metrics["regression"]["mae"]:.4f}  '
          f'MAE(high-Tc)={metrics["regression"]["mae_high_tc"]:.4f}')
    print(f'  Prédictions → {pred_csv}')

In [ ]:
# Tableau comparatif et export CSV
import pandas as pd
from IPython.display import display

rows = []
for mode, ckpt_path, _ in RUNS:
    m = eval_results[mode]
    cls, reg = m['classification'], m['regression']
    rows.append({
        'mode'              : mode,
        'accuracy'          : cls.get('accuracy'),
        'f1'                : cls.get('f1'),
        'roc_auc'           : cls.get('roc_auc'),
        'mae'               : reg.get('mae'),
        'rmse'              : reg.get('rmse'),
        'mae_superconductors': reg.get('mae_superconductors'),
        'mae_high_tc'       : reg.get('mae_high_tc'),
    })

summary = pd.DataFrame(rows)
out_path = PROJECT_DIR / 'colab_ablation_summary.csv'
summary.to_csv(out_path, index=False)

print('=== Résultats test set ===')
display(summary.style.highlight_max(
    subset=['accuracy', 'f1', 'roc_auc'],
    color='lightgreen'
).highlight_min(
    subset=['mae', 'rmse', 'mae_superconductors', 'mae_high_tc'],
    color='lightgreen'
))

if len(summary) == 2:
    d = summary.set_index('mode')
    delta_f1  = d.loc['crys_jepa_dft', 'f1']  - d.loc['dft', 'f1']
    delta_mae = d.loc['dft', 'mae'] - d.loc['crys_jepa_dft', 'mae']
    print(f'\nΔF1  (crys_jepa_dft − dft) = {delta_f1:+.4f}')
    print(f'ΔMAE (dft − crys_jepa_dft) = {delta_mae:+.4f} K')

print(f'\nTableau sauvegardé → {out_path}')

In [ ]:
# Courbes d'apprentissage (val_mae et val_f1 par époque)
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for label, result in [('dft', result_dft), ('crys_jepa_dft', result_jepa)]:
    history = result['history']
    epochs  = [h['epoch'] for h in history]
    val_mae = [h['val']['regression']['mae'] for h in history]
    val_f1  = [h['val']['classification']['f1'] for h in history]
    ax1.plot(epochs, val_mae, label=label)
    ax2.plot(epochs, val_f1,  label=label)

ax1.set_title('Val MAE ↓ par époque')
ax1.set_xlabel('Époque')
ax1.set_ylabel('MAE (K)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title('Val F1 ↑ par époque')
ax2.set_xlabel('Époque')
ax2.set_ylabel('F1')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plot_path = PROJECT_DIR / 'colab_training_curves.png'
fig.savefig(plot_path, dpi=150)
plt.show()
print(f'Figure sauvegardée → {plot_path}')

## 6 · [Phase 3] Ablation multi-seeds — évaluation robuste

Lance 10 runs (2 modes × 5 seeds) pour obtenir des résultats moyenne ± écart-type.

> **Durée estimée** : 5 × ~15 min = **~1h30 sur T4** — s'assurer que la session Colab ne s'interrompt pas.

In [ ]:
# Configuration Phase 3
SEEDS             = [0, 1, 2, 3, 4]
EPOCHS_MULTISEED  = EPOCHS          # Même que l'entraînement principal
MULTISEED_OUT_DIR = str(PROJECT_DIR / 'results/multiseed')

print(f'Seeds       : {SEEDS}')
print(f'Époques/run : {EPOCHS_MULTISEED}')
print(f'Total runs  : {len(SEEDS) * 2}  ({len(SEEDS)} × dft + {len(SEEDS)} × crys_jepa_dft)')
print(f'Output dir  : {MULTISEED_OUT_DIR}')

In [ ]:
# Patcher les configs pour Phase 3 (même device/AMP qu'au-dessus)
import yaml

for name, cfg_path in CONFIGS.items():
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['epochs']      = int(EPOCHS_MULTISEED)
    cfg['training']['batch_size']  = int(BATCH_SIZE)
    cfg['training']['use_amp']     = bool(USE_AMP)
    cfg['training']['num_workers'] = int(NUM_WORKERS)
    cfg['training']['pin_memory']  = DEVICE == 'cuda'
    with cfg_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(f'{name} — device={DEVICE}  AMP={USE_AMP}')

In [ ]:
# Lance l'ablation multi-seeds via le script dédié
seeds_arg = ' '.join(str(s) for s in SEEDS)

result = subprocess.run(
    [
        sys.executable, 'scripts/run_multiseed_ablation.py',
        '--seeds', *[str(s) for s in SEEDS],
        '--output-dir', MULTISEED_OUT_DIR,
    ],
    cwd=PROJECT_DIR,
    text=True,
)
# Les sorties s'affichent en direct (pas de capture) grâce au subprocess sans pipe

In [ ]:
# Afficher les résultats agrégés
import pandas as pd
from IPython.display import display
from pathlib import Path

out = Path(MULTISEED_OUT_DIR)
per_seed = pd.read_csv(out / 'multiseed_per_seed.csv')
summary  = pd.read_csv(out / 'multiseed_summary.csv')

print('=== Résultats par seed ===')
display(per_seed.sort_values(['input_mode', 'seed']))

print('\n=== Résumé stabilité (moyenne ± écart-type sur', len(SEEDS), 'seeds) ===')
KEY = ['f1', 'roc_auc', 'mae', 'mae_high_tc', 'accuracy']
fmt = summary[['input_mode', 'n_seeds']].copy()
for col in KEY:
    m, s = f'{col}_mean', f'{col}_std'
    if m in summary.columns:
        fmt[col] = summary.apply(
            lambda r, m=m, s=s: f"{r[m]:.4f} ± {r[s]:.4f}" if pd.notna(r[m]) else 'N/A',
            axis=1,
        )
display(fmt)

if len(summary) >= 2:
    d = summary.set_index('input_mode')
    if 'crys_jepa_dft' in d.index and 'dft' in d.index:
        delta_f1  = d.loc['crys_jepa_dft', 'f1_mean']  - d.loc['dft', 'f1_mean']
        delta_mae = d.loc['dft', 'mae_mean'] - d.loc['crys_jepa_dft', 'mae_mean']
        std_f1    = (d.loc['crys_jepa_dft', 'f1_std']**2 + d.loc['dft', 'f1_std']**2)**0.5
        print(f'\nΔF1  (crys_jepa_dft − dft) = {delta_f1:+.4f}  (±{std_f1:.4f} pooled std)')
        print(f'ΔMAE (dft − crys_jepa_dft) = {delta_mae:+.4f} K')
        stable_f1 = abs(delta_f1) > 2 * std_f1
        print(f'→ Gain F1 significatif (|Δ| > 2σ) : {"OUI ✓" if stable_f1 else "NON — à confirmer"}')

## Notes

- **Encodeur structurel** : en l'absence d'un checkpoint Crys-JEPA préentraîné, le mode `crys_jepa_dft` utilise le `PlaceholderCrysJEPAEncoder` (MLP léger). Pour brancher un vrai encodeur, renseigner `model.crys_jepa_checkpoint` dans le YAML.
- **AMP** : activé automatiquement sur GPU CUDA (T4/A100), désactivé sur CPU.
- **Persistance** : les checkpoints et CSVs de prédictions sont dans `/content/Crys_JEPA/checkpoints/`. Activer `USE_GDRIVE = True` dans la cellule 0b pour les sauvegarder sur Drive.
- **Quota Colab gratuit** : ~90 min de compute continu. La Phase 3 (10 runs) peut dépasser cette limite — utiliser Colab Pro ou réduire `SEEDS`.
- **Reproduction** : le seed par défaut est fixé à 42 dans les configs ; le multi-seed (Phase 3) varie les seeds de 0 à 4.